# v33 — N0/N1: auxiliary-вес 0,025 вместо 0,25

Один адресный повтор после v32. В **обеих** ветвях изменён только вес дополнительного loss. Те же parent R1, seed, пачки/аугментации, LR, BN, 1600 joint + 200 target-only updates и фиксированный граф.

**Restart Kernel → Run All**: тесты → preflight → короткий smoke → N_ref → N0 → N1 → сравнение с v32 и отчёт. Выбери проектный `.venv`. Лимита времени нет.
Рабочий v25, исходная validation, bbox, порог и завершённый v32 не меняются. N0/N1 стартуют с исходного fold-matched R1, не с финальных v32/smoke-весов.

После прерывания используй тот же RUN_NAME и Run All; replay — максимум последний незавершённый блок 100 updates. Не меняй код/среду между продолжениями. Старый notebook v32 повторно запускать не нужно.


In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до импортов training/ORT
from pathlib import Path
import sys, subprocess, time
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Car-classification-MSK']
ROOT = next((p for p in candidates if (p / 'training/nive_mixed_low_aux.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_NAME = 'low_aux_v1'
DEVICE = 'mps'  # тот же Mac/runtime, что в v32
started = time.perf_counter()
print('Python:', sys.executable, '\nRun:', RUN_NAME, '| device:', DEVICE)


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: low_aux_v1 | device: mps


In [2]:
print('Этап 1/5 — тесты сопоставимости, trainer, resume и метрик', flush=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_nive_low_aux.py', 'tests/test_nive_mixed.py', 'tests/test_nive_transfer.py', 'tests/test_overnight_training.py', 'tests/test_retrieval_policy.py'], check=True, env=os.environ.copy())


Этап 1/5 — тесты сопоставимости, trainer, resume и метрик
........................................................................ [ 55%]
..........................................................               [100%]
=============================== warnings summary ===============================
tests/test_nive_transfer.py::test_real_representation_transfer_and_onnx_export
  /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/training/osnet_ablation_suite.py:547: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
    torch.onnx.export(encoder, torch.zeros(1, 3, variant.size, variant.size), temporary,

tests/test_nive_transf

CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_nive_low_aux.py', 'tests/test_nive_mixed.py', 'tests/test_nive_transfer.py', 'tests/test_overnight_training.py', 'tests/test_retrieval_policy.py'], returncode=0)

In [3]:
from training import nive_mixed_low_aux as experiment
print('Этап 2/5 — сверка с завершённым v32 и защита v25', flush=True)
context = experiment.prepare(RUN_NAME, device=DEVICE)
manifest = context['manifest']
print('Результаты:', context['output'])
print('Auxiliary weight в N0 и N1:', context['plan']['aux_weight'])
print('Main LR:', manifest['config']['encoder_lr'])
print('Parent / data / batches / BN / draws: совпадают с v32')
for arm, coverage in manifest['sampling']['planned_coverage'].items():
    print(arm, '| auxiliary coverage:', f"{coverage['unique_images']}/{coverage['available_images']}")


Этап 2/5 — сверка с завершённым v32 и защита v25
PREFLIGHT: verifying frozen v32, local images and v25 (read-only)


NiVe integrity: train:   0%|          | 0/17070 [00:00<?, ?it/s]

NiVe integrity: test/query:   0%|          | 0/2887 [00:00<?, ?it/s]

NiVe integrity: test/gallery:   0%|          | 0/11778 [00:00<?, ?it/s]

PREFLIGHT OK: same parent, data, batches, LR, BN, draws; auxiliary .025 in BOTH arms
Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_33_nive_low_aux/runs/low_aux_v1
Auxiliary weight в N0 и N1: 0.025
Main LR: 1.1940564013110388e-05
Parent / data / batches / BN / draws: совпадают с v32
N0_target_extra | auxiliary coverage: 4612/4612
N1_nive_mixed | auxiliary coverage: 17070/17070


Аудит похожих изображений переиспользуется из v32 **только после проверки байтов, split и кода аудитора**. Это не новый визуальный осмотр.
Уменьшение auxiliary-loss не уменьшает число обновлений BN на NiVe. BN специально оставлен прежним: не меняем две гипотезы одновременно.
Перед обучением заново извлекается N_ref и проверяется точное совпадение его признаков/метрик с v32. При несовпадении не ослабляй проверку.


In [4]:
print('Этап 3/5 — по три одноразовых update с весом 0.025 и проверка экспорта', flush=True)
smoke = experiment.technical_smoke(context)
print('Smoke:', smoke['status'], '| полный опыт начнётся с исходного R1')


Этап 3/5 — по три одноразовых update с весом 0.025 и проверка экспорта
TRAIN N0_target_extra: 3/3 | target-only | loss=1.2817 | elapsed 00:00:06


/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


TRAIN N1_nive_mixed: 3/3 | target-only | loss=1.2834 | elapsed 00:00:04
Smoke: passed | полный опыт начнётся с исходного R1


In [5]:
print('Этап 4/5 — сопоставимый N0/N1, 1800 updates на ветвь, без лимита времени', flush=True)
results = experiment.run_experiment(context)
print('Статус:', results['status'], '| screening decision:', results['decision'])


Этап 4/5 — сопоставимый N0/N1, 1800 updates на ветвь, без лимита времени
EVAL N_ref: fixed graph=0.844099, raw=0.832842
STAGE N0_target_extra: auxiliary=0.025; new optimizer from original R1
TRAIN N0_target_extra: 25/1800 | joint | loss=1.4261 | elapsed 00:00:50
TRAIN N0_target_extra: 50/1800 | joint | loss=1.4274 | elapsed 00:01:40
TRAIN N0_target_extra: 75/1800 | joint | loss=1.5575 | elapsed 00:02:30
TRAIN N0_target_extra: 100/1800 | joint | loss=1.5145 | elapsed 00:03:20
TRAIN N0_target_extra: 125/1800 | joint | loss=1.4030 | elapsed 00:04:10
TRAIN N0_target_extra: 150/1800 | joint | loss=1.3329 | elapsed 00:05:00
TRAIN N0_target_extra: 175/1800 | joint | loss=1.3471 | elapsed 00:05:50
TRAIN N0_target_extra: 200/1800 | joint | loss=1.4473 | elapsed 00:06:40
TRAIN N0_target_extra: 225/1800 | joint | loss=1.3884 | elapsed 00:07:30
TRAIN N0_target_extra: 250/1800 | joint | loss=1.3352 | elapsed 00:08:20
TRAIN N0_target_extra: 275/1800 | joint | loss=1.3372 | elapsed 00:09:10
TRAIN N0_

In [6]:
from IPython.display import Markdown, display
print('Этап 5/5 — отчёт и сравнение 0.025 против 0.25')
print('Время Run All:', round((time.perf_counter()-started)/60, 1), 'мин')
print('Выбранные updates:', results['selection'])
print('Разница N1 − N0:', results['N1_vs_N0']['delta_map'])
print('Рабочий v25 и предыдущий v32 не изменены.')
display(Markdown((context['output'] / 'REPORT.md').read_text()))


Этап 5/5 — отчёт и сравнение 0.025 против 0.25
Время Run All: 109.1 мин
Выбранные updates: {'N0_target_extra': '0', 'N1_nive_mixed': '0'}
Разница N1 − N0: 0.0
Рабочий v25 и предыдущий v32 не изменены.


# v33 — один N0/N1 повтор: auxiliary 0.025

Статус: complete. Исходная validation и рабочий v25 не изменены.
Parent, seed, данные, порядок пачек/аугментаций, LR, BN, updates и граф сохранены из v32.

| Вариант | Update | Raw mAP | Fixed graph mAP |
|---|---:|---:|---:|
| N_ref | 0 | 0.832842 | 0.844099 |
| N0_target_extra | 0 | 0.832842 | 0.844099 |
| 50% N_ref + 50% N0_target_extra | 0 | 0.832842 | 0.844099 |
| N1_nive_mixed | 0 | 0.832842 | 0.844099 |
| 50% N_ref + 50% N1_nive_mixed | 0 | 0.832842 | 0.844099 |

## Сравнение с первым опытом (0.25)

| Ветка | Старый лучший step | Новый лучший step | Δ single, п.п. | Δ смеси, п.п. |
|---|---:|---:|---:|---:|
| N0_target_extra | 800 | 0 | -0.0262 | -0.1810 |
| N1_nive_mixed | 0 | 0 | +0.0000 | +0.0000 |

## Все заранее назначенные точки

| Ветка | Update | Старый graph (.25) | Новый raw (.025) | Новый graph (.025) |
|---|---:|---:|---:|---:|
| N0_target_extra | 0 | 0.844099 | 0.832842 | 0.844099 |
| N0_target_extra | 400 | 0.841344 | 0.831679 | 0.841184 |
| N0_target_extra | 800 | 0.844361 | 0.832785 | 0.842268 |
| N0_target_extra | 1200 | 0.842637 | 0.832362 | 0.843222 |
| N0_target_extra | 1600 | 0.840464 | 0.834674 | 0.843658 |
| N0_target_extra | 1800 | 0.840066 | 0.833134 | 0.843056 |
| N1_nive_mixed | 0 | 0.844099 | 0.832842 | 0.844099 |
| N1_nive_mixed | 400 | 0.826124 | 0.818517 | 0.831210 |
| N1_nive_mixed | 800 | 0.822410 | 0.819200 | 0.830873 |
| N1_nive_mixed | 1200 | 0.823739 | 0.822664 | 0.834456 |
| N1_nive_mixed | 1600 | 0.821080 | 0.818333 | 0.826202 |
| N1_nive_mixed | 1800 | 0.831807 | 0.830964 | 0.844009 |

Рекомендация screening: **stop**, не разрешение релиза.
Если выбран update 0, этот encoder не обучался на NiVe; его смесь с N_ref не доказывает пользу внешних данных.
Повтор слабее старого N0/его смеси — не новый лучший результат, даже если стал лучше прежнего N1.

Бюджет на ветвь: 1600 joint + 200 target-only updates.
Loss в обеих ветвях: L_main + 0.025 L_aux. Это не доля итогового градиента.
BN-политика не менялась; уменьшение веса loss не уменьшает число обновлений BN на NiVe.
Нормы градиентов — training/*/history.json; per-query AP/top-10 — evaluation/*/metrics.json.
Primary draws разделяют identity. Отбор конкретного checkpoint — адаптивный development, не hidden-test оценка.
Граф/порог/веса смеси не подбирались. Head-free export — исследовательский, без новой candidate-policy.
Это единственная адресная коррекция; нового автоматического поиска и promotion нет.
